# Оценка эффекта воздействия в выбранной модели

В качестве эффекта воздействия рассматривается включение коммунальных услуг в стоимость аренды квартиры

`utilities_included` берем как treatment-переменную

$
utilities\_included =
\begin{cases}
1, & \text{если коммунальные услуги включены в стоимость аренды} \\
0, & \text{иначе}
\end{cases}
$

$
H_1:
\beta_{utilities\_included} > 0
$

то есть включение коммунальных услуг повышает стоимость аренды квартиры

In [1]:
#берем модель 6 из задания 3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
df = pd.read_excel("data.xlsx")
base_vars = [
    'ln_total_meters',
    'rooms_count',
    'center_distance',
    'metro_time_min'
]

building_vars = [
    'relative_floor',
    'is_apartment'
]

rent_condition_vars = [
    'no_commission',
    'has_deposit',
    'utilities_included',
    'kids_allowed',
    'pets_allowed'
]

amenity_vars = [
    'has_furniture',
    'has_washing_machine',
    'has_dryer',
    'has_fridge',
    'has_tv',
    'has_conditioner',
    'has_dishwasher',
    'has_microwave',
    'has_boiler'
]

repair_vars = [
    'repair_designer',
    'repair_euro'
]

base_vars = [x for x in base_vars if x in df.columns]
building_vars = [x for x in building_vars if x in df.columns]
rent_condition_vars = [x for x in rent_condition_vars if x in df.columns]
amenity_vars = [x for x in amenity_vars if x in df.columns]
repair_vars = [x for x in repair_vars if x in df.columns]
model_cols = ['price', 'ln_price'] + base_vars + building_vars + rent_condition_vars + amenity_vars + repair_vars

df_model = df[model_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
df_model['amenities_count'] = df_model[amenity_vars].sum(axis=1)

df_model.shape
formula_m6 = '''
ln_price ~ ln_total_meters
         + rooms_count
         + center_distance
         + relative_floor
         + is_apartment
         + utilities_included
         + repair_euro
         + amenities_count
'''

model_6 = smf.ols(formula_m6, data=df_model).fit()

In [2]:
#считаем эффект
beta = model_6.params['utilities_included']
effect = np.exp(beta) - 1
print("Coefficient:", beta)
print("Treatment effect:", effect)
print("P-value:", model_6.pvalues['utilities_included'])

Coefficient: 0.06465016883593688
Treatment effect: 0.06678576419328208
P-value: 0.09813056230613715


Получаем, эффект воздействия составляет: $e^{0.0647} - 1 \approx 0.0668$
 -квартиры, в стоимость аренды которых включены коммунальные услуги, в среднем стоят примерно на 6.7% дороже при прочих равных условиях.

Полученный эффект статистически значим на 10%-м уровне: $p\text{-value} = 0.098$

Результат в целом ожидаем: включение коммунальных услуг делает условия аренды более удобными для арендаторов, что позволяет арендодателям устанавливать более высокую цену

## Квантильная регрессия

В качестве дополнительной модели строим квантильную регрессию. Ее применение оправдано: рынок аренды жилья является неоднородным, и факторы, влияющие на дешёвые квартиры, могут иначе воздействовать на более дорогие объекты

оценим модели для 25-го квантиля распределения, 50-го, и 75-го

$
H_1:\text{влияние факторов на стоимость аренды различается между ценовыми сегментами рынка}
$


In [3]:
quantiles = [0.25, 0.5, 0.75]
qr_results = {}
for q in quantiles:
    model = smf.quantreg(formula_m6, df_model).fit(q=q)

    qr_results[q] = model

    print("\n" + "="*80)
    print(f"Quantile regression, q = {q}")
    print(model.summary())


Quantile regression, q = 0.25
                         QuantReg Regression Results                          
Dep. Variable:               ln_price   Pseudo R-squared:               0.2909
Model:                       QuantReg   Bandwidth:                      0.1876
Method:                 Least Squares   Sparsity:                        1.044
Date:                Thu, 07 May 2026   No. Observations:                  987
Time:                        09:36:35   Df Residuals:                      978
                                        Df Model:                            8
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              9.8987      0.163     60.878      0.000       9.580      10.218
ln_total_meters        0.2927      0.047      6.242      0.000       0.201       0.385
rooms_count            0.2086      0.029      7.286      0.000      

In [4]:
qr_table = pd.DataFrame({
    q: qr_results[q].params
    for q in quantiles
})

qr_table

,0.25,0.50,0.75
Intercept,9.898681,10.000039,10.211638
ln_total_meters,0.292665,0.331981,0.433816
rooms_count,0.208583,0.227429,0.203929
center_distance,-0.007423,-0.008237,-0.018640
relative_floor,-0.004104,-0.044837,-0.168854
is_apartment,0.185040,0.162650,0.209935
utilities_included,0.058071,0.069375,0.093569
repair_euro,-0.027180,-0.042544,-0.087241
amenities_count,-0.034285,-0.035963,-0.041022


In [5]:
qr_pvalues = pd.DataFrame({
    q: qr_results[q].pvalues
    for q in quantiles
})

qr_pvalues

,0.25,0.50,0.75
Intercept,0.000000e+00,6.863517e-292,1.786044e-204
ln_total_meters,6.435484e-10,5.251008e-10,3.310029e-09
rooms_count,6.576571e-13,6.547338e-13,6.898070e-07
center_distance,1.964788e-03,2.574357e-03,1.032997e-07
relative_floor,9.356215e-01,4.534856e-01,3.465725e-02
is_apartment,1.554360e-04,3.674651e-03,4.586605e-03
utilities_included,8.727142e-02,7.400760e-02,5.772475e-02
repair_euro,4.034670e-01,2.501030e-01,6.420706e-02
amenities_count,8.721386e-06,8.956907e-05,1.023559e-03


Итак, влияние факторов действительно различается для разных ценовых сегментов!

### Площадь квартиры

Коэффициент при переменной `ln_total_meters` увеличивается при переходе к более дорогим квартирам:

| Квантиль | Коэффициент |
|---|---|
| 0.25 | 0.293 |
| 0.50 | 0.332 |
| 0.75 | 0.434 |

Все коэффициенты статистически значимы.

То есть, площадь особенно важна для дорогих квартир

---

### Расстояние до центра

Влияние расстояния до центра также усиливается в верхнем сегменте рынка:

| Квантиль | Коэффициент |
|---|---|
| 0.25 | -0.007 |
| 0.50 | -0.008 |
| 0.75 | -0.019 |

Следовательно, удалённость от центра сильнее снижает стоимость дорогих квартир

---

### Относительный этаж

Для дешёвых и средних квартир переменная `relative_floor` статистически незначима.

Однако для верхнего квантиля коэффициент становится отрицательным и значимым:

$$
\beta = -0.169
$$

Это может означать, что для дорогих квартир расположение на менее удобных этажах сильнее влияет на цену.

---

### Апартаменты

Переменная `is_apartment` остаётся положительной и статистически значимой во всех квантилях

---

### Включённые коммунальные услуги

Коэффициент при `utilities_included` положителен во всех моделях:

| Квантиль | Коэффициент |
|---|---|
| 0.25 | 0.058 |
| 0.50 | 0.069 |
| 0.75 | 0.094 |

Эффект усиливается в верхнем сегменте рынка, включённые коммунальные услуги особенно ценятся в дорогих квартирах.

---



Таким образом, квантильная регрессия подтверждает неоднородность рынка аренды жилья.